In [1]:
!pip -q install transformers
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf datasets huggingface_hub hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.

In [2]:
import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

In [4]:
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "sunny199/iphone5_vlm",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [5]:
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "latex_ocr"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

In [6]:
from datasets import load_dataset
dataset = load_dataset("HuggingFaceM4/ChartQA")
print(dataset)
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
print(train_ds[0])

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…): reconstructing file:   0%|          |  0.00B /  219MB            

data/train-00000-of-00003-49492f364babfa(…): downloading bytes:           |  0.00B            

data/train-00001-of-00003-7302bae5e425bb(…): reconstructing file:   0%|          |  0.00B /  311MB            

data/train-00001-of-00003-7302bae5e425bb(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-e2cd0b7a0f9eb20(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7C391BDD8590>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


In [7]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

    # -----------------------------
    # Load Model
    # -----------------------------
    def load_model(self):
        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth"
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(format_sample, remove_columns=raw.column_names)

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):

        FastVisionModel.for_training(self.model)

        args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(self.model, self.tokenizer),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=args,
        )

        return self

    # -----------------------------
    # Train
    # -----------------------------
    def train(self):
        print("Training for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()

    # -----------------------------
    # Save
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)
        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)
        print("Saved to:", self.cfg.save_dir)

    # -----------------------------
    # Quick Inference
    # -----------------------------
    def quick_infer(self, sample_index=0):

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": self.dataset_info["instruction"]}
            ]}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(self.tokenizer, skip_prompt=True)

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
        )

In [8]:
cfg = VisionFTConfig(
  model_key="qwen2_vl_2b",   # Change model here
  dataset_key="latex_ocr",  # Change dataset here
)

trainer = (
  VisionFineTuner(cfg)
  .load_model()
  .prepare_data()
  .build_trainer()
)

trainer.train()
trainer.save()
trainer.quick_infer()

==((====))==  Unsloth 2026.9.4: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/519 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  344MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Step,Training Loss
10,0.768438
20,0.274155
30,0.195780


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-34/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in vlm_lora_output/tokenizer_config.json.


Saved to: vlm_lora_output


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


\overline { { \cal N } } \overline { { M } } \in { \bf Z } , \overline { { M } } \overline { { P } } \in { \bf Z } , \overline { { P } } \overline { { Q } } \in { \bf Z } .<|im_end|>
